<a href="https://colab.research.google.com/github/hniyongabo/financial_inclusion_summative/blob/main/Financial_Inclusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Financial Inclusion Classifier: Who Has a Bank Account in East Africa?

- **Author:** Harmony Naomi Niyongabo
- **GitHub:** https://github.com/hniyongabo/financial_inclusion_summative
- **Google Drive:** https://drive.google.com/drive/folders/1VNK2eXWbDTxHTHEZS4VMGZGIjMDEuKO6?usp=sharing
- **Video:** [paste link here when done]

This notebook predicts bank account ownership from survey data and compares traditional machine learning with deep learning.




## 1. Download the Dataset

In [2]:
!kaggle datasets download -d dhiebtarak/financial-inclusion-in-africa
!unzip -o financial-inclusion-in-africa.zip

Dataset URL: https://www.kaggle.com/datasets/dhiebtarak/financial-inclusion-in-africa
License(s): unknown
100% 289k/289k [00:00<00:00, 128MB/s]

Archive:  financial-inclusion-in-africa.zip
  inflating: Test.csv                
  inflating: Train.csv               
  inflating: VariableDefinitions.csv  


## 2. Environment & Seed Configuration

This cell imports all the libraries used in the notebook.

In [3]:
import os, random, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import tensorflow as tf

warnings.filterwarnings("ignore")

This cell fixes the random seed to 42 so results repeat on every run.

In [4]:
GLOBAL_SEED = 42

def set_seeds(seed=GLOBAL_SEED):
    """Fix every random number generator so the notebook is reproducible."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)           # python + numpy + tensorflow together
    tf.config.experimental.enable_op_determinism()  # deterministic TF operations

set_seeds()


This cell creates folders for outputs, sets a clean plot style and prints the library versions used, for reproducibility.

In [5]:
for folder in ["models", "figures", "logs", "results"]:
    os.makedirs(folder, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

print("Seed:", GLOBAL_SEED)
print("pandas", pd.__version__, "| numpy", np.__version__,
      "| scikit-learn", sklearn.__version__, "| tensorflow", tf.__version__)

Seed: 42
pandas 2.2.3 | numpy 2.1.3 | scikit-learn 1.6.1 | tensorflow 2.21.0


This cell prints the exact library versions to copy into requirements.txt.

In [6]:
!pip freeze | grep -i -E "^(pandas|numpy|scikit-learn|tensorflow|matplotlib|seaborn|imbalanced-learn|joblib)=="

imbalanced-learn==0.14.2
joblib==1.6.0
matplotlib==3.10.0
numpy==2.1.3
pandas==2.2.3
scikit-learn==1.6.1
seaborn==0.13.2
tensorflow==2.21.0


## 3. Load Dataset

This cell loads the Train dataset using the pandas' `.read_csv` and analyzes dataset information (missing values, various data types, etc) using `.info()`

In [7]:
import pandas as pd

df_raw = pd.read_csv("Train.csv")

df_raw.info()
df_raw.tail()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23524 entries, 0 to 23523
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   country                 23524 non-null  object
 1   year                    23524 non-null  int64 
 2   uniqueid                23524 non-null  object
 3   bank_account            23524 non-null  object
 4   location_type           23524 non-null  object
 5   cellphone_access        23524 non-null  object
 6   household_size          23524 non-null  int64 
 7   age_of_respondent       23524 non-null  int64 
 8   gender_of_respondent    23524 non-null  object
 9   relationship_with_head  23524 non-null  object
 10  marital_status          23524 non-null  object
 11  education_level         23524 non-null  object
 12  job_type                23524 non-null  object
dtypes: int64(3), object(10)
memory usage: 2.3+ MB


,country,year,uniqueid,bank_account,location_type,cellphone_access,household_size,age_of_respondent,gender_of_respondent,relationship_with_head,marital_status,education_level,job_type
23519,Uganda,2018,uniqueid_2113,No,Rural,Yes,4,48,Female,Head of Household,Divorced/Seperated,No formal education,Other Income
23520,Uganda,2018,uniqueid_2114,No,Rural,Yes,2,27,Female,Head of Household,Single/Never Married,Secondary education,Other Income
23521,Uganda,2018,uniqueid_2115,No,Rural,Yes,5,27,Female,Parent,Widowed,Primary education,Other Income
23522,Uganda,2018,uniqueid_2116,No,Urban,Yes,7,30,Female,Parent,Divorced/Seperated,Secondary education,Self employed
23523,Uganda,2018,uniqueid_2117,No,Rural,Yes,10,20,Male,Child,Single/Never Married,Secondary education,No Income


## 4. Data Quality Analysis

This cell checks the dataset size, the ID and year columns, and people with identical answers.

In [8]:
# 1. Size and missing values
print("Rows x columns:", df_raw.shape)
print("Total missing values:", int(df_raw.isna().sum().sum()))

# 2. Identifier and year
print("\nuniqueid:", df_raw["uniqueid"].nunique(), "distinct values in", len(df_raw), "rows\n")
print(pd.crosstab(df_raw["country"], df_raw["year"]))

# 3. Identical feature rows
feature_cols = [c for c in df_raw.columns if c not in ["uniqueid", "year", "bank_account"]]
same = df_raw.duplicated(subset=feature_cols, keep=False)
print(f"\nRows with identical features to another row: {same.sum()} ({same.mean():.1%})")


Rows x columns: (23524, 13)
Total missing values: 0

uniqueid: 8735 distinct values in 23524 rows

year      2016  2017  2018
country                   
Kenya        0     0  6068
Rwanda    8735     0     0
Tanzania     0  6620     0
Uganda       0     0  2101

Rows with identical features to another row: 7497 (31.9%)


This cell finds identical answers that have different bank account labels.



In [9]:
# 1. Conflicting labels
groups = df_raw.groupby(feature_cols)["bank_account"].agg(["nunique", "size"])
conflicts = groups[(groups["size"] > 1) & (groups["nunique"] > 1)]
print(f"Conflicting patterns: {len(conflicts)}, covering {int(conflicts['size'].sum())} rows\n")

Conflicting patterns: 484, covering 1746 rows



This cell counts extreme ages and household sizes

In [10]:
# 2. Outliers (IQR rule)
for col in ["age_of_respondent", "household_size"]:
    s = df_raw[col]
    q1, q3 = s.quantile([0.25, 0.75])
    lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    print(f"{col}: min={s.min()}, median={s.median()}, max={s.max()}, "
          f"outliers={int(((s < lo) | (s > hi)).sum())}")

age_of_respondent: min=16, median=35.0, max=100, outliers=241
household_size: min=1, median=3.0, max=21, outliers=381


This cell lists every category in each text column to find unclear labels.

In [11]:
# 3. Category levels
for col in df_raw.select_dtypes("object").columns.drop(["uniqueid", "bank_account"]):
    print(f"\n{col}\n{df_raw[col].value_counts()}")


country
country
Rwanda      8735
Tanzania    6620
Kenya       6068
Uganda      2101
Name: count, dtype: int64

location_type
location_type
Rural    14343
Urban     9181
Name: count, dtype: int64

cellphone_access
cellphone_access
Yes    17454
No      6070
Name: count, dtype: int64

gender_of_respondent
gender_of_respondent
Female    13877
Male       9647
Name: count, dtype: int64

relationship_with_head
relationship_with_head
Head of Household      12831
Spouse                  6520
Child                   2229
Parent                  1086
Other relative           668
Other non-relatives      190
Name: count, dtype: int64

marital_status
marital_status
Married/Living together    10749
Single/Never Married        7983
Widowed                     2708
Divorced/Seperated          2076
Dont know                      8
Name: count, dtype: int64

education_level
education_level
Primary education                  12791
No formal education                 4515
Secondary education             

## 5. Class Imbalance Check

This cell checks for target proportions using `.value_counts`

In [12]:
# 1. Count of target classes ('No' vs 'Yes')
print("\nTarget counts:")
print(df_raw['bank_account'].value_counts())

# 2. Percentage of target classes
print("\nTarget percentages:")
print(df_raw['bank_account'].value_counts(normalize=True) * 100)


Target counts:
bank_account
No     20212
Yes     3312
Name: count, dtype: int64

Target percentages:
bank_account
No     85.920762
Yes    14.079238
Name: proportion, dtype: float64


## 6. Data Cleaning

We drop `uniqueid` because it repeats across countries, and `year` because it is tied to country. We turn targets into 1 for Yes and 0 for No

In [13]:
df = df_raw.copy() #create a copy
df = df.drop(columns=["uniqueid","year"])
df["bank_account"] = df["bank_account"].map({'Yes': 1, 'No': 0}) #encode with .map

#print columns and target distribution
print("Remaining columns:", df.columns.tolist())
print("\nTarget proportions:")
print(df["bank_account"].value_counts())

Remaining columns: ['country', 'bank_account', 'location_type', 'cellphone_access', 'household_size', 'age_of_respondent', 'gender_of_respondent', 'relationship_with_head', 'marital_status', 'education_level', 'job_type']

Target proportions:
bank_account
0    20212
1     3312
Name: count, dtype: int64


## 7. Feature Engineering

We add two simple columns: mobile_urban (has a phone and lives in a city) and is_working_age (age 25 to 60). We test later whether they help.

In [14]:
# 1. Feature Engineering: demographic, access indicators and working age
df['mobile_urban'] = ((df['cellphone_access'] == 'Yes') & (df['location_type'] == 'Urban')).astype(int)
df['is_working_age'] = ((df['age_of_respondent'] >= 25) & (df['age_of_respondent'] <= 60)).astype(int)

# Verify new columns
print("New shape:", df.shape)
df[['cellphone_access', 'location_type', 'mobile_urban', 'age_of_respondent', 'is_working_age']].head()

New shape: (23524, 13)


,cellphone_access,location_type,mobile_urban,age_of_respondent,is_working_age
0,Yes,Rural,0,24,0
1,No,Rural,0,70,0
2,Yes,Urban,1,26,1
3,Yes,Rural,0,34,1
4,No,Urban,0,26,1


This cell adds an education score from 0 to 4 and a flag for formal employment.

In [15]:
education_order = {
    "No formal education": 0,
    "Primary education": 1,
    "Secondary education": 2,
    "Vocational/Specialised training": 3,
    "Tertiary education": 4,
}
df["education_ord"] = df["education_level"].map(education_order).fillna(1).astype(int)

formal_jobs = ["Formally employed Private", "Formally employed Government"]
df["is_formal_employed"] = df["job_type"].isin(formal_jobs).astype(int)

This cell adds flags for head of household and married status, and lists all new features.

In [16]:
df["is_head"] = (df["relationship_with_head"] == "Head of Household").astype(int)
df["is_married"] = (df["marital_status"] == "Married/Living together").astype(int)

engineered_cols = ["mobile_urban", "is_working_age", "education_ord",
                   "is_formal_employed", "is_head", "is_married"]
print("New shape:", df.shape)

New shape: (23524, 17)


This cell shows how bank account ownership differs across the new features.

In [17]:
for col in ["mobile_urban", "is_working_age", "is_formal_employed", "is_head", "is_married"]:
    rates = df.groupby(col)["bank_account"].mean().round(3)
    print(f"{col}: no={rates.loc[0]}, yes={rates.loc[1]}")

print("\neducation_ord:")
print(df.groupby("education_ord")["bank_account"].mean().round(3))

mobile_urban: no=0.099, yes=0.252
is_working_age: no=0.089, yes=0.166
is_formal_employed: no=0.111, yes=0.604
is_head: no=0.097, yes=0.177
is_married: no=0.113, yes=0.174

education_ord:
education_ord
0    0.039
1    0.086
2    0.233
3    0.570
4    0.511
Name: bank_account, dtype: float64


## 8. Train / Validation / Test Split

We split the data into three parts: 70% for training, 15% for validation, and 15% for testing. Class proportions are the same across all three parts so the test would not be biased.

In [18]:
from sklearn.model_selection import train_test_split

# separate features and target
X = df.drop(columns=['bank_account'])
y = df['bank_account']

# Set aside 15% of the data for testing
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=GLOBAL_SEED, stratify=y
    )
# Split the remaining 85% into training (70%) and validation (15%)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.1765, random_state=GLOBAL_SEED, stratify=y_train_val
)

# Print how many rows are in each part
print("Train set size:", len(X_train))
print("Validation set size:", len(X_val))
print("Test set size:", len(X_test))


Train set size: 16465
Validation set size: 3530
Test set size: 3529


This cell checks that train, validation and test each have the same share of account holders.

In [19]:
for name, labels in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(f"{name}: {len(labels)} rows, share with an account = {labels.mean():.4f}")

print("Number of features:", X_train.shape[1])

train: 16465 rows, share with an account = 0.1408
validation: 3530 rows, share with an account = 0.1408
test: 3529 rows, share with an account = 0.1408
Number of features: 16


## 9. Preprocessing Pipeline

This cell defines a function that prepares the data, with or without the new features.

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols = ["household_size", "age_of_respondent"]
cat_cols = ["country", "location_type", "cellphone_access", "gender_of_respondent",
            "relationship_with_head", "marital_status", "education_level", "job_type"]
flag_cols = ["mobile_urban", "is_working_age", "is_formal_employed", "is_head", "is_married"]

def make_preprocessor(use_engineered):
    num = num_cols + (["education_ord"] if use_engineered else [])
    parts = [("num", StandardScaler(), num),
             ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)]
    if use_engineered:
        parts.append(("flags", "passthrough", flag_cols))
    return ColumnTransformer(parts)

This cell prepares both versions of the data, learning from the training rows only.

In [21]:
prep_raw = make_preprocessor(use_engineered=False)
X_train_raw = prep_raw.fit_transform(X_train)
X_val_raw, X_test_raw = prep_raw.transform(X_val), prep_raw.transform(X_test)

prep_eng = make_preprocessor(use_engineered=True)
X_train_eng = prep_eng.fit_transform(X_train)
X_val_eng, X_test_eng = prep_eng.transform(X_val), prep_eng.transform(X_test)

print("Original features:", X_train_raw.shape[1], ", With engineered:", X_train_eng.shape[1])

Original features: 39 , With engineered: 45


This cell gives all "don't know" answers one clear label, Unknown

In [22]:
unclear = {
    "marital_status": ["Dont know"],
    "education_level": ["Other/Dont know/RTA"],
    "job_type": ["Dont Know/Refuse to answer"],
}
for col, labels in unclear.items():
    df[col] = df[col].replace(labels, "Unknown")
    print(col, ":", int((df[col] == "Unknown").sum()), "rows labelled Unknown")

marital_status : 8 rows labelled Unknown
education_level : 35 rows labelled Unknown
job_type : 126 rows labelled Unknown


## 10. Evaluation Setup

This cell creates the experiment log that every result in the report will come from.

In [23]:
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, precision_recall_curve)

results_log = []

def log_run(run_id, phase, model_name, settings, features, y_val, val_prob,
            y_train=None, train_prob=None, threshold=0.5, split="70/15/15", notes=""):
    results_log[:] = [r for r in results_log if r["Run"] != run_id]
    pred = (val_prob >= threshold).astype(int)
    row = {"Run": run_id, "Phase": phase, "Model": model_name, "Settings": settings,
           "Features": features, "Split": split, "Threshold": round(threshold, 3),
           "Precision": round(precision_score(y_val, pred), 4),
           "Recall": round(recall_score(y_val, pred), 4),
           "F1": round(f1_score(y_val, pred), 4),
           "PR-AUC": round(average_precision_score(y_val, val_prob), 4),
           "ROC-AUC": round(roc_auc_score(y_val, val_prob), 4)}
    if train_prob is not None:
        train_auc = roc_auc_score(y_train, train_prob)
        row["Train ROC-AUC"] = round(train_auc, 4)
        row["AUC gap"] = round(train_auc - row["ROC-AUC"], 4)
    row["Notes"] = notes
    results_log.append(row)
    return row

This cell adds a function to save the results table and one to pick the best threshold.

In [24]:
def save_results():
    table = pd.DataFrame(results_log)
    order = table["Run"].str.extract(r"(\d+)")[0].astype(int)
    table = table.assign(_n=order).sort_values("_n").drop(columns="_n")
    table = table[[c for c in table.columns if c != "Notes"] + ["Notes"]]
    table.to_csv("results/experiment_results_table.csv", index=False)
    return table.reset_index(drop=True)

def best_threshold(y_true, prob):
    p, r, t = precision_recall_curve(y_true, prob)
    f1 = 2 * p[:-1] * r[:-1] / (p[:-1] + r[:-1] + 1e-12)
    return float(t[f1.argmax()])

## 11. Experiments: Traditional ML

### Feature ablation (E1 and E2)

This cell defines a reusable function that trains, scores and logs every traditional model.

In [25]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate

cv_split = StratifiedKFold(n_splits=5, shuffle=True, random_state=GLOBAL_SEED)

def run_sklearn(run_id, phase, name, settings, clf, use_engineered, notes="", cv=True):
    model = Pipeline([("prep", make_preprocessor(use_engineered)), ("clf", clf)])
    start = time.time()
    model.fit(X_train, y_train)
    fit_s = round(time.time() - start, 1)
    train_prob = model.predict_proba(X_train)[:, 1]
    val_prob = model.predict_proba(X_val)[:, 1]
    feats = "original + engineered" if use_engineered else "original"
    row = log_run(run_id, phase, name, settings, feats, y_val, val_prob,
                  y_train, train_prob, notes=notes)
    row["Fit seconds"] = fit_s
    if cv:
        s = cross_validate(model, X_train, y_train, cv=cv_split, n_jobs=-1,
                           scoring=["roc_auc", "average_precision"])
        row["CV ROC-AUC"] = f"{s['test_roc_auc'].mean():.4f} ± {s['test_roc_auc'].std():.4f}"
        row["CV PR-AUC"] = f"{s['test_average_precision'].mean():.4f} ± {s['test_average_precision'].std():.4f}"
    print(f"{run_id} {name}: F1={row['F1']}, PR-AUC={row['PR-AUC']}, ROC-AUC={row['ROC-AUC']}, fit {fit_s}s")
    return model

Experiment 1 trains a logistic regression on the original columns only, as a baseline.

In [27]:
from sklearn.linear_model import LogisticRegression

lr_raw = run_sklearn("E1", "Features", "Logistic Regression", "C=1, class_weight=balanced",
                     LogisticRegression(class_weight="balanced", max_iter=1000, random_state=GLOBAL_SEED),
                     use_engineered=False, notes="Baseline on the 10 original columns")

E1 Logistic Regression: F1=0.4997, PR-AUC=0.5445, ROC-AUC=0.8454, fit 0.7s


Experiment 2 repeats Experiment 1 with the new features added, to test if they help.

In [28]:
lr_eng = run_sklearn("E2", "Features", "Logistic Regression", "C=1, class_weight=balanced",
                     LogisticRegression(class_weight="balanced", max_iter=1000, random_state=GLOBAL_SEED),
                     use_engineered=True, notes="Same model, original + 6 engineered columns")

E2 Logistic Regression: F1=0.5024, PR-AUC=0.547, ROC-AUC=0.8479, fit 0.5s


This cell saves the results table and shows the two experiments side by side.

In [29]:
save_results()[["Run", "Features", "F1", "PR-AUC", "ROC-AUC", "AUC gap", "CV ROC-AUC", "CV PR-AUC"]]

,Run,Features,F1,PR-AUC,ROC-AUC,AUC gap,CV ROC-AUC,CV PR-AUC
0,E1,original,0.4997,0.5445,0.8454,0.0075,0.8503 ± 0.0095,0.5631 ± 0.0215
1,E2,original + engineered,0.5024,0.5470,0.8479,0.0065,0.8517 ± 0.0098,0.5657 ± 0.0238


### Experiment 3: Histogram-Based Gradient Boosting Classifier
We apply gradient-boosted decision trees that sequentially correct residual errors from preceding estimators. Histogram binning speeds up continuous feature evaluations while built-in class weighting preserves sensitivity toward positive bank holders.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

# Train sequential gradient boosting trees
model_hgb = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    class_weight='balanced',
    random_state=GLOBAL_SEED
)
model_hgb.fit(X_train_proc, y_train)

# Evaluate on validation data
y_val_pred_hgb = model_hgb.predict(X_val_proc)
y_val_prob_hgb = model_hgb.predict_proba(X_val_proc)[:, 1]

evaluate_performance("Exp 3: HistGradientBoosting (LR 0.05)", y_val, y_val_pred_hgb, y_val_prob_hgb)

### Experiment 4: Regularized and Fine-Tuned Gradient Boosting
We optimize the boosting model by reducing the learning rate and introducing L2 leaf regularization to reduce variance. These adjustments penalize overly complex splits and improve generalization.

In [ ]:
# Fine-tune tree depth and regularization parameters
model_hgb_tuned = HistGradientBoostingClassifier(
    learning_rate=0.03,
    max_iter=300,
    min_samples_leaf=30,
    l2_regularization=1.0,
    class_weight='balanced',
    random_state=GLOBAL_SEED
)
model_hgb_tuned.fit(X_train_proc, y_train)

# Evaluate on validation data
y_val_pred_hgb_tuned = model_hgb_tuned.predict(X_val_proc)
y_val_prob_hgb_tuned = model_hgb_tuned.predict_proba(X_val_proc)[:, 1]

evaluate_performance("Exp 4: Tuned HistGradientBoosting (LR 0.03, Reg 1.0)", y_val, y_val_pred_hgb_tuned, y_val_prob_hgb_tuned)

### TensorBoard Logging Setup and Loss Weight Calculation
A local directory is initialized to record TensorBoard event logs for tracking training metrics across epochs. Balanced loss weights are calculated so the model is penalized for misclassifications on minority bank account holders more heavily than majority non-holders.

In [ ]:
import os
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Create TensorBoard log folder
log_dir = "logs/fit"
os.makedirs(log_dir, exist_ok=True)

# Compute loss weights for class 0 and class 1
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights_dict = {0: float(weights[0]), 1: float(weights[1])}

print("Calculated Class Weights:", class_weights_dict)

### Experiment 5: Baseline Multi-Layer Perceptron Neural Network
A feedforward neural network with two hidden layers and ReLU activations establishes the deep learning baseline. Training progress is monitored via TensorBoard, using the computed inverse class weights to handle class imbalance.

In [ ]:
tf.keras.utils.set_random_seed(GLOBAL_SEED)

# Define baseline architecture
model_mlp = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_proc.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model_mlp.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[tf.keras.metrics.AUC(name='auc')]
)

The cell below trains MLP baseline neural network and saves it to Tensorboard

In [ ]:
# Attach TensorBoard logging callback
tb_mlp = tf.keras.callbacks.TensorBoard(log_dir=f"{log_dir}/exp5_baseline_mlp")

# Train the baseline network
model_mlp.fit(
    X_train_proc, y_train,
    validation_data=(X_val_proc, y_val),
    epochs=25,
    batch_size=64,
    class_weight=class_weights_dict,
    callbacks=[tb_mlp],
    verbose=0
)

# Evaluate on validation data
y_val_prob_mlp = model_mlp.predict(X_val_proc, verbose=0).flatten()
y_val_pred_mlp = (y_val_prob_mlp >= 0.5).astype(int)

evaluate_performance("Exp 5: Baseline MLP (Dense 64-32)", y_val, y_val_pred_mlp, y_val_prob_mlp)


### Experiment 6: Regularized Deep MLP with Dropout and Early Stopping
Dropout layers are used to reduce overfitting on the data. An early stopping callback ends training when validation loss stops improving and restores the best model weights.




In [ ]:
tf.keras.utils.set_random_seed(GLOBAL_SEED)

# Define regularized architecture
model_mlp_reg = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_proc.shape[1],)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model_mlp_reg.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='binary_crossentropy',
    metrics=[tf.keras.metrics.AUC(name='auc')]
)

# Define callbacks for logging and overfitting control
tb_mlp_reg = tf.keras.callbacks.TensorBoard(log_dir=f"{log_dir}/exp6_regularized_mlp")
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)


The cell below trains the MLP and uses Dropout and Early Stopping. Additionally, the model is evaluated on validation data

In [ ]:

# Train the regularized network
model_mlp_reg.fit(
    X_train_proc, y_train,
    validation_data=(X_val_proc, y_val),
    epochs=40,
    batch_size=64,
    class_weight=class_weights_dict,
    callbacks=[tb_mlp_reg, early_stop],
    verbose=0
)

# Evaluate on validation data
y_val_prob_mlp_reg = model_mlp_reg.predict(X_val_proc, verbose=0).flatten()
y_val_pred_mlp_reg = (y_val_prob_mlp_reg >= 0.5).astype(int)

evaluate_performance("Exp 6: Regularized MLP (Dropout + EarlyStop)", y_val, y_val_pred_mlp_reg, y_val_prob_mlp_reg)

### Results Table

In [ ]:
import pandas as pd

# Create structured DataFrame from recorded runs
summary_df = pd.DataFrame(results_log)

# Sort by ROC-AUC to rank best performing models
summary_df = summary_df.sort_values(by="ROC-AUC", ascending=False).reset_index(drop=True)

# Save to CSV for submission
summary_df.to_csv("experiment_results_table.csv", index=False)

# Display styled dataframe
summary_df.style.set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-weight', 'bold')]},
    {'selector': 'td', 'props': [('text-align', 'center')]}
])

### TensorBoard Interactive Dashboard
The TensorBoard extension is initialized to visually inspect training dynamics across the deep learning runs. Loss and area under the ROC curve (AUC) trajectories are displayed across epochs for both baseline and regularized multi-layer perceptron models.

In [ ]:
# Load the TensorBoard notebook extension
%load_ext tensorboard

# Launch TensorBoard pointing to our saved logs folder
%tensorboard --logdir logs/fit

The cell below zips tensorboard logs

In [ ]:
import shutil

# Compress the entire logs directory into a zip file
shutil.make_archive("tensorboard_logs", 'zip', "logs")


The cell below copies logs to the Drive folder "Financial_Inclusion_Assignment"

In [ ]:
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

# Copy zip and CSV into a designated Drive folder
target_folder = '/content/drive/MyDrive/Financial_Inclusion_Assignment/'
os.makedirs(target_folder, exist_ok=True)

shutil.copy('tensorboard_logs.zip', target_folder)
shutil.copy('experiment_results_table.csv', target_folder)

print(f"Successfully copied to: {target_folder}")